# Notebook 13 — Stability

**Continual Learning in Context: RML Extension for CL-BENCH**

Notebook 13 asks:

> Once useful context is discovered, does it persist across context changes?

Previous notebooks established:

- **00** — context and baseline workflow
- **01** — gain as measurable context signal
- **07** — latent structures that make context reusable
- **11** — stateful vs. stateless comparison

Notebook 13 focuses on **stability**:

> useful structure retained across variant boundaries.

Outputs:

- `results/13_stability_by_variant.csv`
- `results/13_boundary_transfer.csv`
- `results/13_context_stability.csv`
- `results/13_stability_summary.json`
- `figures/13_stability_by_variant.png`
- `figures/13_boundary_transfer.png`
- `figures/13_stability_decay.png`
- `figures/13_context_stability_map.png`
- `results/13_stability_artifacts.zip`

## 0. Bootstrap Repo Path

This cell works locally or in Colab.

If opened directly in Colab and the repo is missing, it clones:

```text
https://github.com/thinkthoughts/continual-learning-bench-rml
```

In [ ]:
from pathlib import Path
import sys
import os
import subprocess

REPO_URL = "https://github.com/thinkthoughts/continual-learning-bench-rml.git"
REPO_NAME = "continual-learning-bench-rml"

def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

def find_rml_root(start: Path) -> Path | None:
    start = start.resolve()
    for base in [start, *start.parents]:
        if (base / "src" / "gain.py").exists():
            return base
        if (base / "rml" / "src" / "gain.py").exists():
            return base / "rml"
        if (base / REPO_NAME / "rml" / "src" / "gain.py").exists():
            return base / REPO_NAME / "rml"
    return None

cwd = Path.cwd().resolve()
RML_ROOT = find_rml_root(cwd)

if RML_ROOT is None and running_in_colab():
    repo_path = Path("/content") / REPO_NAME

    if repo_path.exists():
        print("Repo already exists. Pulling latest changes...")
        subprocess.run(["git", "-C", str(repo_path), "pull"], check=False)
    else:
        print("Cloning repo...")
        subprocess.run(["git", "clone", REPO_URL, str(repo_path)], check=True)

    RML_ROOT = repo_path / "rml"
    os.chdir(RML_ROOT)

elif RML_ROOT is not None:
    os.chdir(RML_ROOT)

else:
    raise RuntimeError(
        "Could not locate rml/src/gain.py. "
        "Run this notebook inside the repo, or in Colab allow the bootstrap cell to clone the repo."
    )

if str(RML_ROOT) not in sys.path:
    sys.path.insert(0, str(RML_ROOT))

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Running in Colab:", running_in_colab())
print("Current working directory:", Path.cwd())
print("RML root:", RML_ROOT)
print("results:", RESULTS_DIR)
print("figures:", FIGURES_DIR)

## 1. Imports

In [ ]:
import json
import zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.gain import compute_gain
from src.stability import compute_stability, compute_plasticity

print("Imports complete.")

## 2. Load Prior Trajectory

Notebook 13 prefers outputs from Notebook 11:

```text
results/11_state_vs_stateless_instances.csv
```

Fallback order:

1. `results/11_state_vs_stateless_instances.csv`
2. `results/01_gain_signal_trajectory.csv`
3. `results/00_context_gain.csv`
4. synthetic toy trajectory

In [ ]:
trajectory_11 = RESULTS_DIR / "11_state_vs_stateless_instances.csv"
trajectory_01 = RESULTS_DIR / "01_gain_signal_trajectory.csv"
trajectory_00 = RESULTS_DIR / "00_context_gain.csv"

if trajectory_11.exists():
    df = pd.read_csv(trajectory_11)
    source_file = trajectory_11
    print("Loaded:", trajectory_11)
elif trajectory_01.exists():
    df = pd.read_csv(trajectory_01)
    source_file = trajectory_01
    print("Loaded:", trajectory_01)
elif trajectory_00.exists():
    df = pd.read_csv(trajectory_00)
    source_file = trajectory_00
    print("Loaded:", trajectory_00)
else:
    print("No prior trajectory found. Creating fallback toy trajectory.")
    df = pd.DataFrame({
        "instance": np.arange(1, 13),
        "variant": [
            "A", "A", "A", "A",
            "B", "B", "B", "B",
            "C", "C", "C", "C",
        ],
        "stateful_reward": [
            0.18, 0.22, 0.28, 0.35,
            0.43, 0.48, 0.46, 0.52,
            0.40, 0.45, 0.55, 0.60,
        ],
        "stateless_reward": [
            0.18, 0.19, 0.20, 0.21,
            0.22, 0.20, 0.21, 0.22,
            0.23, 0.22, 0.24, 0.25,
        ],
    })
    source_file = None

if "gain" not in df.columns:
    if "state_advantage" in df.columns:
        df["gain"] = df["state_advantage"]
    else:
        df["gain"] = compute_gain(
            df["stateful_reward"].tolist(),
            df["stateless_reward"].tolist(),
        )

if "state_advantage" not in df.columns:
    df["state_advantage"] = df["gain"]

if "gain_delta" not in df.columns:
    df["gain_delta"] = df["gain"].diff().fillna(0.0)

if "cumulative_gain" not in df.columns:
    df["cumulative_gain"] = df["gain"].cumsum()

df

## 3. Define Stability

In this notebook, stability means:

> useful context remains useful after context transitions.

Operationally, we inspect:

1. **Gain at variant boundaries**
2. **Boundary transfer ratio**
3. **Gain decay or recovery after transitions**
4. **Instance-level stability states**

A high stability signal means accumulated context survives a context shift.

## 4. Variant Boundaries

A variant boundary is the first instance of a new variant.

These boundaries are where accumulated context is tested most clearly.

In [ ]:
variant_order = (
    df[["variant", "instance"]]
    .drop_duplicates("variant")
    .sort_values("instance")
)

boundary_instances = variant_order["instance"].tolist()
boundary_indices = [
    int(df.index[df["instance"] == inst][0])
    for inst in boundary_instances
]

variant_boundaries = boundary_indices

boundary_table = pd.DataFrame({
    "variant": variant_order["variant"].tolist(),
    "boundary_instance": boundary_instances,
    "zero_based_index": boundary_indices,
})

boundary_table

## 5. Stability by Variant

For each variant, we compare:

- first gain,
- last gain,
- mean gain,
- gain change across the variant.

This gives a first-pass stability profile.

In [ ]:
variant_rows = []

for variant, group in df.groupby("variant", sort=False):
    first_gain = float(group["gain"].iloc[0])
    last_gain = float(group["gain"].iloc[-1])
    mean_gain = float(group["gain"].mean())
    gain_change = last_gain - first_gain

    if gain_change > 0.02:
        profile = "strengthening"
    elif gain_change < -0.02:
        profile = "weakening"
    else:
        profile = "stable"

    variant_rows.append({
        "variant": variant,
        "instances": int(len(group)),
        "first_instance": int(group["instance"].iloc[0]),
        "last_instance": int(group["instance"].iloc[-1]),
        "first_gain": first_gain,
        "last_gain": last_gain,
        "mean_gain": mean_gain,
        "gain_change": float(gain_change),
        "stability_profile": profile,
    })

stability_by_variant = pd.DataFrame(variant_rows)
stability_by_variant

## 6. Boundary Transfer

Boundary transfer compares the first gain after a context change to the previous gain before the boundary.

\[
\text{transfer ratio}
=
\frac{g_{after}}{g_{before}}
\]

Interpretation:

- ratio near 1: context transferred smoothly
- ratio below 1: transfer cost or context revision
- ratio above 1: new context improved quickly

In [ ]:
boundary_rows = []

ordered = df.sort_values("instance").reset_index(drop=True)

for i, row in boundary_table.iterrows():
    boundary_instance = int(row["boundary_instance"])
    variant = row["variant"]

    current_idx = int(ordered.index[ordered["instance"] == boundary_instance][0])

    if current_idx == 0:
        before_gain = np.nan
        after_gain = float(ordered.loc[current_idx, "gain"])
        transfer_ratio = np.nan
        transfer_cost = 0.0
        label = "initial_context"
    else:
        before_gain = float(ordered.loc[current_idx - 1, "gain"])
        after_gain = float(ordered.loc[current_idx, "gain"])
        transfer_ratio = (
            after_gain / before_gain
            if before_gain != 0
            else np.nan
        )
        transfer_cost = max(0.0, before_gain - after_gain)

        if transfer_cost > 0.08:
            label = "revision_cost"
        elif transfer_ratio >= 1:
            label = "smooth_or_improved_transfer"
        else:
            label = "partial_transfer"

    boundary_rows.append({
        "to_variant": variant,
        "boundary_instance": boundary_instance,
        "gain_before_boundary": before_gain,
        "gain_at_boundary": after_gain,
        "transfer_ratio": transfer_ratio,
        "transfer_cost": float(transfer_cost),
        "boundary_label": label,
    })

boundary_transfer = pd.DataFrame(boundary_rows)
boundary_transfer

## 7. Stability Decay

This section measures gain as a function of distance from the nearest variant boundary.

If useful context stabilizes, gain should recover or increase as the system adapts within a variant.

In [ ]:
decay_df = ordered.copy()

distance_values = []

for _, row in decay_df.iterrows():
    inst = int(row["instance"])
    variant = row["variant"]
    boundary_inst = int(
        boundary_table.loc[
            boundary_table["variant"] == variant,
            "boundary_instance"
        ].iloc[0]
    )
    distance_values.append(inst - boundary_inst)

decay_df["distance_from_variant_boundary"] = distance_values

stability_decay = (
    decay_df.groupby("distance_from_variant_boundary")
    .agg(
        mean_gain=("gain", "mean"),
        min_gain=("gain", "min"),
        max_gain=("gain", "max"),
        instances=("instance", "count"),
    )
    .reset_index()
)

stability_decay

## 8. Context Stability States

Each instance is assigned a simple state:

- **stable**: gain positive and not declining
- **declining**: gain positive but lower than previous instance
- **recovering**: gain increases after a decline
- **neutral**: gain is zero
- **unstable**: gain is negative

In [ ]:
state_labels = []

for i, row in ordered.iterrows():
    gain = float(row["gain"])
    delta = float(row["gain_delta"])

    if gain < 0:
        label = "unstable"
    elif gain == 0:
        label = "neutral"
    elif delta >= 0:
        label = "stable"
    elif delta < 0:
        label = "declining"
    else:
        label = "unknown"

    state_labels.append(label)

context_stability = ordered.copy()
context_stability["stability_state"] = state_labels

context_stability[[
    "instance",
    "variant",
    "gain",
    "gain_delta",
    "stability_state",
]]

## 9. Figure — Stability by Variant

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(
    stability_by_variant["variant"],
    stability_by_variant["mean_gain"],
)

ax.axhline(0, linewidth=1)
ax.set_title("Notebook 13: Mean Stability Signal by Variant")
ax.set_xlabel("Variant")
ax.set_ylabel("Mean Gain")
ax.grid(True, axis="y", alpha=0.3)

stability_variant_path = FIGURES_DIR / "13_stability_by_variant.png"
fig.tight_layout()
fig.savefig(stability_variant_path, dpi=160)

stability_variant_path

## 10. Figure — Boundary Transfer

In [ ]:
plot_boundary = boundary_transfer.dropna(subset=["transfer_ratio"]).copy()

fig, ax = plt.subplots(figsize=(8, 5))

if len(plot_boundary) > 0:
    labels = [
        f"{row['to_variant']}\ninst {int(row['boundary_instance'])}"
        for _, row in plot_boundary.iterrows()
    ]

    ax.bar(labels, plot_boundary["transfer_ratio"])
    ax.axhline(1.0, linewidth=1, linestyle="--")
else:
    ax.text(0.5, 0.5, "No non-initial boundaries found", ha="center", va="center")

ax.set_title("Notebook 13: Boundary Transfer Ratio")
ax.set_xlabel("Boundary")
ax.set_ylabel("Gain After / Gain Before")
ax.grid(True, axis="y", alpha=0.3)

boundary_transfer_path = FIGURES_DIR / "13_boundary_transfer.png"
fig.tight_layout()
fig.savefig(boundary_transfer_path, dpi=160)

boundary_transfer_path

## 11. Figure — Stability Decay

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    stability_decay["distance_from_variant_boundary"],
    stability_decay["mean_gain"],
    marker="o",
)

ax.axhline(0, linewidth=1)
ax.set_title("Notebook 13: Stability Across Distance From Boundary")
ax.set_xlabel("Distance From Variant Boundary")
ax.set_ylabel("Mean Gain")
ax.grid(True, alpha=0.3)

stability_decay_path = FIGURES_DIR / "13_stability_decay.png"
fig.tight_layout()
fig.savefig(stability_decay_path, dpi=160)

stability_decay_path

## 12. Figure — Context Stability Map

In [ ]:
state_counts = (
    context_stability
    .groupby(["variant", "stability_state"])
    .size()
    .unstack(fill_value=0)
)

preferred_order = ["stable", "declining", "recovering", "neutral", "unstable"]
for col in preferred_order:
    if col not in state_counts.columns:
        state_counts[col] = 0

state_counts = state_counts[preferred_order]

fig, ax = plt.subplots(figsize=(9, 5))

bottom = np.zeros(len(state_counts))

for column in state_counts.columns:
    values = state_counts[column].values
    ax.bar(state_counts.index, values, bottom=bottom, label=column)
    bottom += values

ax.set_title("Notebook 13: Context Stability States by Variant")
ax.set_xlabel("Variant")
ax.set_ylabel("Instance Count")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

context_stability_path = FIGURES_DIR / "13_context_stability_map.png"
fig.tight_layout()
fig.savefig(context_stability_path, dpi=160)

context_stability_path

## 13. Summary

In [ ]:
basic_stability = compute_stability(
    ordered["gain"].tolist(),
    variant_boundaries,
)

basic_plasticity = compute_plasticity(
    ordered["gain"].tolist(),
    variant_boundaries,
)

largest_transfer_cost_row = boundary_transfer.loc[
    boundary_transfer["transfer_cost"].idxmax()
].to_dict()

summary = {
    "total_instances": int(len(ordered)),
    "variants": ordered["variant"].drop_duplicates().tolist(),
    "trajectory_source": str(source_file) if source_file is not None else "fallback",
    "mean_gain": float(ordered["gain"].mean()),
    "total_gain": float(ordered["gain"].sum()),
    "basic_stability_boundary_gain": float(basic_stability),
    "basic_plasticity_non_boundary_gain": float(basic_plasticity),
    "largest_transfer_cost_variant": str(largest_transfer_cost_row["to_variant"]),
    "largest_transfer_cost_instance": int(largest_transfer_cost_row["boundary_instance"]),
    "largest_transfer_cost": float(largest_transfer_cost_row["transfer_cost"]),
    "stability_state_counts": {
        state: int(count)
        for state, count in context_stability["stability_state"].value_counts().items()
    },
}

summary

## 14. Export Artifacts

In [ ]:
stability_by_variant_path = RESULTS_DIR / "13_stability_by_variant.csv"
boundary_transfer_csv_path = RESULTS_DIR / "13_boundary_transfer.csv"
context_stability_csv_path = RESULTS_DIR / "13_context_stability.csv"
stability_decay_csv_path = RESULTS_DIR / "13_stability_decay.csv"
summary_path = RESULTS_DIR / "13_stability_summary.json"
zip_path = RESULTS_DIR / "13_stability_artifacts.zip"

stability_by_variant.to_csv(stability_by_variant_path, index=False)
boundary_transfer.to_csv(boundary_transfer_csv_path, index=False)
context_stability.to_csv(context_stability_csv_path, index=False)
stability_decay.to_csv(stability_decay_csv_path, index=False)

summary_with_metadata = {
    **summary,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "13_stability.ipynb",
    "extension": "continual-learning-bench-rml",
    "repo": REPO_URL,
}

with open(summary_path, "w") as f:
    json.dump(summary_with_metadata, f, indent=2)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in [
        stability_by_variant_path,
        boundary_transfer_csv_path,
        context_stability_csv_path,
        stability_decay_csv_path,
        summary_path,
        stability_variant_path,
        boundary_transfer_path,
        stability_decay_path,
        context_stability_path,
    ]:
        z.write(path, arcname=path.name)

print("Saved artifacts:")
for path in [
    stability_by_variant_path,
    boundary_transfer_csv_path,
    context_stability_csv_path,
    stability_decay_csv_path,
    summary_path,
    stability_variant_path,
    boundary_transfer_path,
    stability_decay_path,
    context_stability_path,
    zip_path,
]:
    print("-", path)

## 15. Download Artifacts

In Colab, this downloads the zip.

Locally, the archive is saved under:

```text
rml/results/13_stability_artifacts.zip
```

In [ ]:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Download helper is only active in Colab.")
    print("Artifact archive:", zip_path)

## 16. Notebook 13 Claim

Gain measures whether context helped.

State advantage measures whether accumulated state contributed.

Stability measures whether useful structure persists across context changes.

\[
\text{gain}
\rightarrow
\text{state advantage}
\rightarrow
\text{stability}
\]

In RML terms:

> Continual learning requires preserving reusable structure without freezing stale assumptions.

Notebook 17 will next examine **plasticity**:

> Can the system adapt to new evidence while retaining useful structure?